In [1]:
import anndata as ad
import scanpy as sc


This data is the most straightforward to read, as its in a simple h5ad file

In [2]:
filePath = "../rawFiles/Kanemaru/Global_raw.h5ad"
adata = sc.read_h5ad(filePath)

Theres multiple cell types in the heart tissue so we're only grabbing those that are identified as cardiomyocyte.
Also, I've found that larger donor size causes much higher variability in the cells. This is fine but needs to be offset with more cells. However, more cells means more ram usage, and I'm already cutting it close. Additionally, more cells means that when we filter for the most variable features, its going to skew more towards variability between these cells, rather than between these cells and hipsc-cms, which we do not want.

In [3]:
adata = adata[adata.obs["scANVI_predictions"].isin(["Ventricular Cardiomyocyte", "Atrial Cardiomyocyte"])].copy()
adata = adata[adata.obs["donor"].isin(["D1", "D2", "D3"])].copy()

removing cells that don't express much (probably damaged), and genes that are not present in many cells (probably not there) This is also where we would normally filter by mitochondrial expression, BUT this dataset already came with it filtered. Kinda annoying because it meant I had to follow the same 20% removal for every other dataset for consistency.

In [4]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

Randomly grabbing 10000 of the cells, to cut down on how many we have for both memory conservation, and keeping it the same size as the other samples. the other bits of the code here are for removing any batch that has fewer than 1000 cells in it. I've found that every so often, the random subset creates a batch that has a small amount of cells, and it causes batch correction to freak out and fail, so this is a safeguard from that.

In [5]:
sc.pp.subsample(adata, n_obs=10000)
batch_counts = adata.obs["batch"].value_counts()
keep_batches = batch_counts[batch_counts >= 1000].index
adata = adata[adata.obs["batch"].isin(keep_batches)].copy()

In [6]:
# Mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# Ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# Hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains(r"^HB[ABDEGMQZ]\d*(?!\w)")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
    )

Saving the un-normalized/log-ed data to "counts" for latter batch correction that expects raw count data.

In [7]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [8]:
adata.obs["source"] = "Kanemaru"
adata.obs.rename(columns = {"scANVI_predictions" : "condition"}, inplace = True)
adata.obs["system"] = "ATCM"

adata.obs["Day"] = "Cardiomyocyte"

In [9]:
adata.write_zarr("ProcessedData/KanemaruScanpy.zarr")

/home/kai/Documents/SCTLM-Bridges/.venv/lib/python3.14/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


In [10]:
adata = ad.read_zarr("ProcessedData/KanemaruScanpy.zarr")
